# Visualise disagreements

We want to calculate measures of overall agreement or disagreement, and
pairwise agreement measures (differences between experts).

Patterns of disagreement: frequency of agreement of all experts, three
experts or only two experts, etc.

We can look at the problem from three angles:

- for each *input type*, how many expert assign the same
  group/membership? what is the disagreement between experts?
- for each *experts*, how many types do they assign to each group
- 

## Set up

### Load libraries

In [ ]:
library(readr)
library(readxl)
library(dplyr)
library(tidyr)
library(stringr)
library(ggplot2)


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


### Relative paths

In [ ]:
here::i_am("read-input-xref-data.ipynb")

here() starts at /Users/elliotgould/Documents/GitHub/TASVEG-XREF

## Read data from all experts

In the previous section we combined data from four different experts.

In [ ]:
xref_table_experts <- read_csv(here::here("output-XREFs","First-round-output.csv"))

Let’s see this full table:

In [4]:
xref_table_experts

# A tibble: 965 × 4 
 VEGCODE efg memb expert
 <chr> <chr> <dbl> <chr> 
 1 AAP TF1.7 1 AK 
 2 AAP TF1.7 1 AL 
 3 AAP TF1.7 1 JB 
 4 AAP TF1.7 0.7 MGT 
 5 AAP TF1.6 0.3 MGT 
 6 AHF F2.2 0.4 AK 
 7 AHF F2.3 0.4 AK 
 8 AHF TF1.3 0.2 AK 
 9 AHF F2.2 0.333 AL 
 10 AHF F2.3 0.4 AL 
 # ℹ 955 more rows 
 # ℹ Use `print(n = ...)` to see more rows

## Summarise the data

Now that we have the input from four experts in the same format, we can
produce some informative summaries:

In [18]:
(candidates_per_vegcode <- xref_table_experts |>
  group_by(VEGCODE) |>
  summarise(`Nr. of candidates`=n_distinct(efg), `Highest membership`=max(memb)))

# A tibble: 156 × 3 
 VEGCODE `Nr. of candidates` `Highest membership`
 <chr> <int> <dbl> 
 1 AAP 2 1 
 2 AHF 4 0.7
 3 AHL 5 1 
 4 AHS 1 1 
 5 ARS 3 1 
 6 ASF 2 1 
 7 ASP 1 1 
 8 ASS 1 1 
 9 AUS 2 1 
 10 AWU 5 1 
 # ℹ 146 more rows 
 # ℹ Use `print(n = ...)` to see more rows

For example, how many candidate functional groups do we have per type
(identified by `VEGCODE`)

In [6]:
candidates_per_vegcode |>
  pull(`Nr. of candidates`) |>
  table()


 1  2  3  4  5  9 
35 68 31 13  8  1 

In [7]:
xref_table_experts |>
  group_by(VEGCODE, efg, memb) |>
  summarise(`Nr. of experts`=n_distinct(expert), 
    experts=paste(expert,collapse=";"),
    .groups="drop") |>
  arrange(VEGCODE, desc(`Nr. of experts`), desc(memb))

# A tibble: 631 × 5 
 VEGCODE efg memb `Nr. of experts` experts 
 <chr> <chr> <dbl> <int> <chr> 
 1 AAP TF1.7 1 3 AK;AL;JB
 2 AAP TF1.7 0.7 1 MGT 
 3 AAP TF1.6 0.3 1 MGT 
 4 AHF F2.3 0.4 2 AK;AL 
 5 AHF F2.3 0.7 1 MGT 
 6 AHF F2.3 0.6 1 JB 
 7 AHF F2.2 0.4 1 AK 
 8 AHF F2.2 0.333 1 AL 
 9 AHF TF1.3 0.333 1 AL 
 10 AHF TF1.3 0.3 1 MGT 
 # ℹ 621 more rows 
 # ℹ Use `print(n = ...)` to see more rows

In [8]:
xref_table_experts |>
  group_by(efg, expert) |>
  summarise(`types`=n_distinct(VEGCODE), 
    `membership total`=sum(memb),
    .groups="drop") 

# A tibble: 131 × 4 
 efg expert types `membership total`
 <chr> <chr> <int> <dbl> 
 1 F1.1 AK 1 0.1 
 2 F1.1 AL 1 0.111
 3 F1.1 JB 1 0.1 
 4 F1.2 AK 1 0.1 
 5 F1.2 AL 1 0.111
 6 F1.2 JB 1 0.2 
 7 F1.4 AK 1 0.1 
 8 F1.4 AL 1 0.111
 9 F1.5 AK 1 0.1 
 10 F1.5 AL 1 0.111
 # ℹ 121 more rows 
 # ℹ Use `print(n = ...)` to see more rows

Create the heatmap plot

This plot can reveal errors in nomenclature (“TF7.2”, presumably
refering to “T7.2”) or genuine differences between experts (only one
expert assigning “TF1.5” or “TF1.4”)

In [ ]:
mtz <- xref_table_experts |>
  group_by(efg, expert) |>
  summarise(`membership`=sum(memb, na.rm=TRUE),
    .groups="drop") |>
  pivot_wider(names_from = "efg", values_from="membership")

#vegdist(mtz)

In [ ]:
dissim_matrix <- dist(mtz[,-1], method = "euclidean")

In [ ]:
plot(hclust(dissim_matrix,),
 labels=mtz$expert ,
     main = "Dissimilarity Between Experts", 
     xlab = "Experts", 
     ylab = "Distance")

## Pairwise diferences in membership

In [ ]:
mtz <- pivot_wider(
  xref_table_experts,
  names_from = "expert", values_from="memb") %>%
    replace(is.na(.),0)
mtz


# A tibble: 365 × 6 
 VEGCODE efg AK AL JB MGT
 <chr> <chr> <dbl> <dbl> <dbl> <dbl> 
 1 AAP TF1.7 1 1 1 0.7
 2 AAP TF1.6 0 0 0 0.3
 3 AHF F2.2 0.4 0.333 0.25 0 
 4 AHF F2.3 0.4 0.4 0.6 0.7
 5 AHF TF1.3 0.2 0.333 0.05 0.3
 6 AHF F3.1 0 0 0.1 0 
 7 AHL TF1.3 1 1 0.1 0 
 8 AHL TF1.4 0 0 0.55 0 
 9 AHL TF1.7 0 0 0.3 0 
 10 AHL MFT1.3 0 0 0.05 0 
 # ℹ 355 more rows 
 # ℹ Use `print(n = ...)` to see more rows

In [14]:

  mtz |>
    filter(AK>0 | JB>0) |>
    summarise(mean(AK-JB))

# A tibble: 1 × 1 
 `mean(AK - JB)`
 <dbl> 
 1 - 0 . 000 1 6 9

In [15]:
mtz |>
    filter(AK>0 | MGT>0) |>
    mutate(disagreement=abs(AK-MGT)) |>
    group_by(efg) |>
    summarise(
      n=n_distinct(VEGCODE),
      disagreement=mean(disagreement)) |>
    arrange(desc(disagreement))

# A tibble: 37 × 3 
 efg n disagreement
 <chr> <int> <dbl> 
 1 F3.3 1 1 
 2 TF1.3 7 0.707
 3 F2.3 6 0.617
 4 T3.4 1 0.5 
 5 MT1.1 1 0.4 
 6 T3.2 31 0.4 
 7 T7.5 10 0.39 
 8 F2.2 2 0.35 
 9 MFT1.3 5 0.35 
 10 T2.5 32 0.328
 # ℹ 27 more rows 
 # ℹ Use `print(n = ...)` to see more rows

In [ ]:
xref_expert_ALL <- read_csv(here::here("input-XREFs/GET_Level3_LUT_120826.csv"))

In [ ]:
efg_cols <- colnames(xref_expert_ALL)[grepl("[MFT0-9]+.[0-9]+",colnames(xref_expert_ALL))]
pivot_longer(xref_expert_ALL,cols=any_of(efg_cols),names_to="efg",values_to="memb") |>
  transmute(VEGCODE=TASVEG, efg, memb, expert="aggregated") |>
  filter(!is.na(memb))

# A tibble: 403 × 4 
 VEGCODE efg memb expert 
 <chr> <chr> <dbl> <chr> 
 1 AAP F2.8 0.04 aggregated
 2 AAP TF1.6 0.02 aggregated
 3 AAP TF1.7 0.94 aggregated
 4 AHF F2.2 0.35 aggregated
 5 AHF F2.3 0.4 aggregated
 6 AHF F2.4 0.06 aggregated
 7 AHF F3.1 0.02 aggregated
 8 AHF TF1.3 0.17 aggregated
 9 AHL F2.3 0.06 aggregated
 10 AHL TF1.3 0.642 aggregated
 # ℹ 393 more rows 
 # ℹ Use `print(n = ...)` to see more rows